**Three Outliers | 2023**
# Penyisihan Dataquest UNAIR: Prediksi curah hujan pada IKN Nusantara 

## Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import catboost as cb
import xgboost as xgb
import lightgbm as lgb

import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

DATA_DIR = "../data"
SUBMISSION_DIR = "../submissions"

## Load Data

In [2]:
ori_train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
ori_train_df.head()

,datetime,datetime_iso,time-zone,temp,visibility,d_point,feels,min_temp,max_temp,prssr,sea_level,grnd_level,hum,wind_spd,wind_deg,rain_1h,rain_3h,snow_1h,snow_3h,clouds
0,283996800,1979-01-01 00:00:00+00:00,28800,24.75 Celcius,NaN,23.89 C,25.76 C,24.28,25.22°C,1012,undetermined,NaN,95,0.82,320.0 °,zero,0,NaN,NaN,100
1,284000400,1979-01-01 01:00:00+00:00,28800,24.58 C,NaN,23.73 C,25.57 C,23.99 C,25.26 C,1012,NaN,NaN,95,0.96 m/s,338.0°,0,0,0,0,100
2,284004000,1979-01-01 02:00:00+00:00,28800,26.6 Celcius,unidentified,24.06 C,26.6 C,26.1 C,27.39,1012,NaN,undetermined,86,1.22 m/s,339.0°,0,volume:zero,NaN,NaN,99
3,284007600,1979-01-01 03:00:00+00:00,28800,27.31 Celcius,NaN,24.37 C,30.9 C,26.59,28.36 C,1012,NaN,undetermined,84,1.08 m/s,342,0.13,nol,0,NaN,94
4,284011200,1979-01-01 04:00:00+00:00,28800,27.41,NaN,25.05 C,31.54 C,26.58 C,28.31 °C,1011,NaN,undetermined,87,0.86,336.0°,0.34,nol,NaN,0,100


In [3]:
ori_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 341880 entries, 0 to 341879
Data columns (total 20 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   datetime      341880 non-null  int64 
 1   datetime_iso  341880 non-null  object
 2   time-zone     341880 non-null  int64 
 3   temp          341880 non-null  object
 4   visibility    51112 non-null   object
 5   d_point       341880 non-null  object
 6   feels         341880 non-null  object
 7   min_temp      341880 non-null  object
 8   max_temp      341880 non-null  object
 9   prssr         341880 non-null  object
 10  sea_level     192964 non-null  object
 11  grnd_level    192919 non-null  object
 12  hum           341880 non-null  object
 13  wind_spd      341880 non-null  object
 14  wind_deg      341880 non-null  object
 15  rain_1h       341880 non-null  object
 16  rain_3h       192329 non-null  object
 17  snow_1h       192696 non-null  object
 18  snow_3h       192699 non

## Preprocessing

In [24]:
train_df = ori_train_df.copy()

### Handle Missing Value

In [5]:
train_df.isnull().sum()

datetime             0
datetime_iso         0
time-zone            0
temp                 0
visibility      290768
d_point              0
feels                0
min_temp             0
max_temp             0
prssr                0
sea_level       148916
grnd_level      148961
hum                  0
wind_spd             0
wind_deg             0
rain_1h              0
rain_3h         149551
snow_1h         149184
snow_3h         149181
clouds               0
dtype: int64

In [6]:
train_df.isnull().sum() / len(train_df) * 100

datetime         0.000000
datetime_iso     0.000000
time-zone        0.000000
temp             0.000000
visibility      85.049725
d_point          0.000000
feels            0.000000
min_temp         0.000000
max_temp         0.000000
prssr            0.000000
sea_level       43.557974
grnd_level      43.571136
hum              0.000000
wind_spd         0.000000
wind_deg         0.000000
rain_1h          0.000000
rain_3h         43.743711
snow_1h         43.636364
snow_3h         43.635486
clouds           0.000000
dtype: float64

Columns that has missing value: 
- visibility: 290768 (85%)
- sea_level: 148916 (43.5%)
- grnd_level: 148961 (43.5%)
- rain_3h: 149551 (43.7%)
- snow_1h: 149184 (43.6%)
- snow_3h: 149181 (43.6%)


In [7]:
train_df['visibility'].value_counts()

unrecognized    4352
-1m             4349
unknown         4326
-1km            4278
unidentified    4263
-1 km           4263
-1              4257
                4241
undefined       4238
-1 m            4234
missing         4165
empty           4146
Name: visibility, dtype: int64

In [8]:
train_df['sea_level'].value_counts()

undetermined     154582
-1                 4890
unestablished      4823
unsettled          4809
not_recorded       4804
not-recorded       4803
unknown            4792
not recorded       4735
unspecified        4726
Name: sea_level, dtype: int64

In [9]:
train_df['grnd_level'].value_counts()

undetermined     148814
not recorded       5615
not_recorded       5579
unspecified        5537
-1                 5511
unsettled          5497
not-recorded       5480
unknown            5443
unestablished      5443
Name: grnd_level, dtype: int64

### Clean Columns

In [10]:
train_df['time-zone'].unique()

array([28800], dtype=int64)

In [11]:
train_df.temp.unique()

array(['24.75 Celcius', '24.58 C', '26.6 Celcius', ..., '136.06 °C',
       '33.48 Celcius', '34.32 C'], dtype=object)

In [12]:
print(train_df['clouds'].apply(
    lambda x: float(re.search(r'\d+(\.\d+)?', x).group(0)) if re.search(r'\d+(\.\d+)?', x) else None).isnull().sum())
train_df['clouds'].apply(
    lambda x: float(re.search(r'\d+(\.\d+)?', x).group(0)) if re.search(r'\d+(\.\d+)?', x) else None)

0


0         100.0
1         100.0
2          99.0
3          94.0
4         100.0
          ...  
341875     99.0
341876    100.0
341877     97.0
341878     98.0
341879     99.0
Name: clouds, Length: 341880, dtype: float64

In [13]:
train_df[train_df['rain_1h'].apply(lambda x: not x[0].isdigit())]['rain_1h']

0         zero
17        -1mm
20            
36        zero
44        -1mm
          ... 
341820        
341827        
341856    zero
341865    -1mm
341876        
Name: rain_1h, Length: 32040, dtype: object

In [14]:
train_df.snow_1h.unique()

array([nan, '0', 'nol', 'volume:zero', '0mm', '0 mm', 'volume:0', 'zero',
       '0 milimeter', 'no-snow', 'no_snow'], dtype=object)

In [15]:
train_df.snow_3h.unique()

array([nan, '0', 'no-snow', '0 mm', 'zero', 'volume:0', '0mm', 'no_snow',
       'volume:zero', '0 milimeter', 'nol'], dtype=object)

In [27]:
def clean_cols(df, is_train=True):
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s')

    # function to clean unit
    def clean_unit(x):
        temp_pattern = r'-?\d+(\.\d+)?'
        regex = re.search(temp_pattern, str(x))
        return float(regex.group(0)) if regex else np.nan

    # clean unit
    df['temp'] = df['temp'].apply(clean_unit)
    df['d_point'] = df['d_point'].apply(clean_unit)
    df['feels'] = df['feels'].apply(clean_unit)
    df['min_temp'] = df['min_temp'].apply(clean_unit)
    df['max_temp'] = df['max_temp'].apply(clean_unit)
    df['prssr'] = df['prssr'].apply(clean_unit)
    df['hum'] = df['hum'].apply(clean_unit) / 100
    df['wind_spd'] = df['wind_spd'].apply(clean_unit)
    df['wind_deg'] = df['wind_deg'].apply(clean_unit)
    df['clouds'] = df['clouds'].apply(clean_unit) / 100
    
    df.loc[df['hum'] > 1, 'hum'] = 1

    # clean rain_1h
    if is_train:
        df.loc[df['rain_1h'] == 'zero', 'rain_1h'] = 0
        df.loc[df['rain_1h'] == ' ', 'rain_1h'] = np.nan
        df.dropna(subset=['rain_1h'], inplace=True)
        df['rain_1h'] = df['rain_1h'].apply(clean_unit)
        df.loc[df['rain_1h'] < 0, 'rain_1h'] = 0

    # drop useless columns
    df.drop(['datetime_iso',
             'time-zone',
             'visibility',
             'sea_level',
             'grnd_level',
             'rain_3h',
             'snow_1h',
             'snow_3h'],
            axis=1, inplace=True)

    return df

In [28]:
train_df = clean_cols(train_df)
train_df.head()

,datetime,temp,d_point,feels,min_temp,max_temp,prssr,hum,wind_spd,wind_deg,rain_1h,clouds
0,1979-01-01 00:00:00,24.75,23.89,25.76,24.28,25.22,1012.0,0.0095,0.82,320.0,0.00,0.0100
1,1979-01-01 01:00:00,24.58,23.73,25.57,23.99,25.26,1012.0,0.0095,0.96,338.0,0.00,0.0100
2,1979-01-01 02:00:00,26.60,24.06,26.60,26.10,27.39,1012.0,0.0086,1.22,339.0,0.00,0.0099
3,1979-01-01 03:00:00,27.31,24.37,30.90,26.59,28.36,1012.0,0.0084,1.08,342.0,0.13,0.0094
4,1979-01-01 04:00:00,27.41,25.05,31.54,26.58,28.31,1011.0,0.0087,0.86,336.0,0.34,0.0100


In [29]:
train_df.isnull().sum()

datetime    0
temp        0
d_point     0
feels       0
min_temp    0
max_temp    0
prssr       0
hum         0
wind_spd    0
wind_deg    0
rain_1h     0
clouds      0
dtype: int64

## Feature Engineering

In [36]:
train_df['year'] = train_df['datetime'].dt.year
train_df['month'] = train_df['datetime'].dt.month
train_df['week'] = train_df['datetime'].dt.week
train_df['day'] = train_df['datetime'].dt.day
train_df['hour'] = train_df['datetime'].dt.hour
train_df['dayofweek'] = train_df['datetime'].dt.dayofweek

## Pre-Modeling

In [37]:
X = train_df.drop(['rain_1h'], axis=1)
y = train_df['rain_1h']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(256472, 17) (256472,) (64118, 17) (64118,)


In [39]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [40]:
# normalize data
scaler = StandardScaler()
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## Modeling

In [122]:
kf = KFold(n_splits=5)
rmse_all_model = {
    # 'ridge': [],
    'rfr': [],
    'cbr': [],
    'xgbr': [],
    'lgbmr': []
}
for i, (train_index, test_index) in enumerate(kf.split(X_train)):
    X_train_fold = X_train.values[train_index]
    y_train_fold = y_train.values[train_index]
    X_test_fold = X_train.values[test_index]
    y_test_fold = y_train.values[test_index]

    # model_1 = Ridge().fit(X_train_fold[1:], y_train_fold)
    model_2 = RandomForestRegressor().fit(X_train_fold, y_train_fold)
    model_3 = cb.CatBoostRegressor(verbose=False).fit(X_train_fold, y_train_fold)
    model_4 = xgb.XGBRegressor().fit(X_train_fold, y_train_fold)
    model_5 = lgb.LGBMRegressor().fit(X_train_fold, y_train_fold)

    # mse_model_1 = rmse(y_test_fold, model_1.predict(X_test_fold))
    mse_model_2 = rmse(y_test_fold, model_2.predict(X_test_fold))
    mse_model_3 = rmse(y_test_fold, model_3.predict(X_test_fold))
    mse_model_4 = rmse(y_test_fold, model_4.predict(X_test_fold))
    mse_model_5 = rmse(y_test_fold, model_5.predict(X_test_fold))

    # rmse_all_model['ridge'].append(mse_model_1)
    rmse_all_model['rfr'].append(mse_model_2)
    rmse_all_model['cbr'].append(mse_model_3)
    rmse_all_model['xgbr'].append(mse_model_4)
    rmse_all_model['lgbmr'].append(mse_model_5)

for key, value in rmse_all_model.items():
    print(f'{key} rmse: {np.mean(value)}')

rfr rmse: 0.9012060143565286
cbr rmse: 0.8866767098323336
xgbr rmse: 0.888490263625622
lgbmr rmse: 0.8898898679938739


In [41]:
cbr_model = cb.CatBoostRegressor(random_state=42)
cbr_model.fit(X_train, y_train)

Learning rate set to 0.098358
0:	learn: 0.9482571	total: 16.8ms	remaining: 16.8s
1:	learn: 0.9342843	total: 32.6ms	remaining: 16.3s
2:	learn: 0.9222126	total: 48.6ms	remaining: 16.1s
3:	learn: 0.9117537	total: 67ms	remaining: 16.7s
4:	learn: 0.9026583	total: 83.8ms	remaining: 16.7s
5:	learn: 0.8949581	total: 100ms	remaining: 16.6s
6:	learn: 0.8890188	total: 116ms	remaining: 16.5s
7:	learn: 0.8833077	total: 136ms	remaining: 16.9s
8:	learn: 0.8782120	total: 153ms	remaining: 16.8s
9:	learn: 0.8741482	total: 184ms	remaining: 18.2s
10:	learn: 0.8703834	total: 208ms	remaining: 18.7s
11:	learn: 0.8671413	total: 227ms	remaining: 18.7s
12:	learn: 0.8647874	total: 245ms	remaining: 18.6s
13:	learn: 0.8625669	total: 263ms	remaining: 18.5s
14:	learn: 0.8604554	total: 279ms	remaining: 18.3s
15:	learn: 0.8584915	total: 295ms	remaining: 18.2s
16:	learn: 0.8566621	total: 313ms	remaining: 18.1s
17:	learn: 0.8555017	total: 329ms	remaining: 18s
18:	learn: 0.8540732	total: 347ms	remaining: 17.9s
19:	learn:

In [72]:
y_pred = cbr_model.predict(X_test)
print(rmse(y_test, y_pred))

0.8085103302549229


In [34]:
# tanpa fe tahun, hari, dll
y_pred = cbr_model.predict(X_test)
print(rmse(y_test, y_pred))

0.8777809235774906


In [58]:
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
test_df = clean_cols(test_df, is_train=False)
test_df.head()

,datetime,temp,d_point,feels,min_temp,max_temp,prssr,hum,wind_spd,wind_deg,clouds
0,2018-01-01 00:00:00,26.59,23.66,26.59,26.02,27.16,1009.0,0.84,1.45,355.0,0.97
1,2018-01-01 01:00:00,26.51,24.92,26.51,26.06,28.04,1009.0,0.91,1.67,351.0,0.95
2,2018-01-01 02:00:00,28.68,25.71,34.68,28.03,29.30,1009.0,0.84,1.72,345.0,0.90
3,2018-01-01 03:00:00,28.84,25.25,34.51,28.52,29.08,1008.0,0.81,1.49,339.0,0.91
4,2018-01-01 04:00:00,29.75,24.62,35.38,29.31,30.57,1007.0,0.74,1.39,339.0,0.96


In [59]:
test_df.isnull().sum()

datetime    0
temp        0
d_point     1
feels       0
min_temp    0
max_temp    0
prssr       0
hum         0
wind_spd    0
wind_deg    0
clouds      0
dtype: int64

In [60]:
test_df.loc[test_df.d_point.isnull(), 'd_point'] = test_df['temp']
test_df.isnull().sum()

datetime    0
temp        0
d_point     0
feels       0
min_temp    0
max_temp    0
prssr       0
hum         0
wind_spd    0
wind_deg    0
clouds      0
dtype: int64

In [61]:
test_df['year'] = test_df['datetime'].dt.year
test_df['month'] = test_df['datetime'].dt.month
test_df['week'] = test_df['datetime'].dt.week
test_df['day'] = test_df['datetime'].dt.day
test_df['hour'] = test_df['datetime'].dt.hour
test_df['dayofweek'] = test_df['datetime'].dt.dayofweek

In [62]:
test_df[num_cols] = scaler.transform(test_df[num_cols])

y_tes_pred = cbr_model.predict(test_df)
submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
submission['rain_1h'] = y_tes_pred
submission.loc[submission['rain_1h'] < 0, 'rain_1h'] = 0
submission.to_csv(f"{SUBMISSION_DIR}/submission_baseline1.csv", index=False)

### Pycaret

In [65]:
from pycaret.regression import *

In [66]:
s_regressor = setup(
    data=train_df,
    target="rain_1h",
    session_id=42,
)

In [70]:
train_df.to_csv(f"{DATA_DIR}/train_cleaned.csv", index=False)

In [69]:
best_model = compare_models()

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,23:27:59
Status,. . . . . . . . . . . . . . . . . .,Loading Dependencies
Estimator,. . . . . . . . . . . . . . . . . .,Compiling Library
